# 🎯 04 — Model Validation & Sensitivity Analysis (ROC-AUC / SPSA)
> **Validation statistique par points d'eau réels (WPDx) et analyse de sensibilité**

Ce notebook évalue quantitativement la capacité prédictive du modèle par courbe ROC et score AUC, puis calcule la sensibilité spatiale de chaque critère (SPSA et MRSA).


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(ROOT))

from src.hydromap.validation import compute_roc_auc, single_parameter_sensitivity, map_removal_sensitivity

print('[OK] Modules de validation importés.')


## 1. Validation de la capacité discriminante (Courbe ROC & AUC)


In [ ]:
# Données de forages de contrôle : 1 = productif (débit > 2.5 m3/h), 0 = sec / faible débit
np.random.seed(42)
n_wells = 120
y_true = np.random.choice([0, 1], size=n_wells, p=[0.4, 0.6])
# Les forages productifs ont tendance à avoir un score GWPI plus élevé
y_scores = np.where(y_true == 1, np.random.beta(5, 2, size=n_wells), np.random.beta(2, 4, size=n_wells))

fpr, tpr, auc = compute_roc_auc(y_true, y_scores)
print(f"Score AUC obtenu : {auc:.3f}")

# Tracé de la courbe ROC
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='#1a9850', lw=2.5, label=f'Modèle GWPI (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], color='#888888', linestyle='--', label='Hasard (AUC = 0.500)')
plt.xlabel('Taux de faux positifs (FPR)', fontsize=11)
plt.ylabel('Taux de vrais positifs (TPR)', fontsize=11)
plt.title('Courbe ROC — Validation du potentiel par forages de terrain', fontsize=12, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 2. Analyse de sensibilité à paramètre unique (SPSA)


In [ ]:
mask = np.ones((50, 50), dtype=bool)
layers = {
    'geology': np.random.uniform(0.3, 0.9, (50, 50)),
    'rainfall': np.random.uniform(0.1, 0.8, (50, 50)),
    'slope': np.random.uniform(0.4, 0.95, (50, 50)),
    'tpi': np.random.uniform(0.3, 0.7, (50, 50)),
}
weights = {'geology': 0.438, 'rainfall': 0.267, 'slope': 0.147, 'tpi': 0.147}

spsa = single_parameter_sensitivity(layers, weights, mask)
print('Poids effectifs moyens (SPSA) :')
for k, eff_w in spsa.items():
    theo_w = weights[k] * 100.0
    print(f'  • {k:<10} : Poids théorique = {theo_w:.1f}% | Poids effectif = {eff_w:.1f}%')
